In [78]:
import pandas as pd

from xgboost import XGBClassifier, XGBRegressor

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import *
from itertools import combinations
from scipy.stats import uniform, truncnorm, randint
from xgboost import plot_importance

from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor, BaggingRegressor, \
    ExtraTreesRegressor, VotingRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor, BayesianRidge, \
    HuberRegressor, PassiveAggressiveRegressor, RANSACRegressor, TheilSenRegressor, TweedieRegressor, PoissonRegressor, \
    GammaRegressor, ARDRegression, LassoLars, OrthogonalMatchingPursuit, LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.svm import LinearSVR, NuSVR, SVR
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.isotonic import IsotonicRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.dummy import DummyRegressor

from lightgbm import LGBMRegressor

from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler, label_binarize


In [ ]:
def data_load_2(file_name, classifier_col, cols_to_drop, debug=False, scalar=1, make_graphs=False, train_data=True):
    df = pd.read_csv(str(file_name), delimiter=',')
    row_count = df.shape[0]  # gives number of row count
    col_count = df.shape[1]  # gives number of col count

    if debug:
        print("Rows:", row_count)
        print("Columns:", col_count, )

    if train_data:
        y = df[
            
            
        ]
        X = df.drop(columns=[classifier_col])
    else:
        X = df.copy()

    if classifier_col == 'ACTM_value':
        X = X.drop(['ACT_value'], axis=1)
    elif classifier_col == 'ACT_value':
        X = X.drop(['ACTM_value'], axis=1)

    for col in cols_to_drop:
        if col in X.columns:
            X = X.drop(columns=[col])
        else:
            print(f"Drop miss: {col!r}")

    if 'Zip Code' in X.columns:
        X = X.drop(columns=['Zip Code'])

    X_obj = X.loc[:, X.dtypes == object]
    X_nobj = X.loc[:, X.dtypes != object]

    print(X_nobj.columns)

    if not X_nobj.empty:
        if scalar == 0:  # No scaling
            X_s = X_nobj
        if scalar == 1:  # Min_Max_Scaler
            min_max_scalar = MinMaxScaler()
            x_scaled = min_max_scalar.fit_transform(X_nobj)
            X_s = pd.DataFrame(x_scaled, columns=X_nobj.columns)
        elif scalar == 2:
            standard_scalar = StandardScaler()
            x_scaled = standard_scalar.fit_transform(X_nobj)
            X_s = pd.DataFrame(x_scaled)

    if not X_obj.empty:
        X_obj_OH = pd.get_dummies(X_obj)
        X = pd.concat([X_obj_OH, X_nobj], axis=1, sort=False)
        if not X_nobj.empty:
            X_s = pd.concat([X_obj_OH, X_s], axis=1, sort=False)
        else:
            X_s = X

    bad_cols = X_s.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    if bad_cols:
        raise ValueError(f"Non-numeric columns remain: {bad_cols}")

    if train_data:
        # Split dataset into training set (70%) and test set (30%)
        X_train, X_test, y_train, y_test = train_test_split(X_s, y, test_size=0.3, random_state=1, stratify=y, shuffle=True)

        return X_train, X_test, y_train, y_test
    else:
        return X_s

def data_load(file_name, classifier_col, cols_to_drop, debug=False, scalar=1, make_graphs=False, train_data=True):
    df = pd.read_csv(file_name, delimiter=',')
    df.columns = df.columns.str.strip()

    if train_data:
        y = df[classifier_col]
        X = df.drop(columns=[classifier_col])
    else:
        X = df.copy()

    if classifier_col == 'ACTM_value' and 'ACT_value' in X.columns:
        X = X.drop(columns=['ACT_value'])
    elif classifier_col == 'ACT_value' and 'ACTM_value' in X.columns:
        X = X.drop(columns=['ACTM_value'])

    for col in cols_to_drop:
        if col in X.columns:
            X = X.drop(columns=[col])
        else:
            print(f"drop miss: " + col)

    if 'Zip Code' in X.columns:
        X = X.drop(columns=['Zip Code'])

    X_obj = X.select_dtypes(include=['object', 'string', 'category'])
    X_nobj = X.select_dtypes(exclude=['object', 'string', 'category'])

    if not X_obj.empty:
        X_obj_OH = pd.get_dummies(X_obj, dummy_na=False)
    else:
        X_obj_OH = pd.DataFrame(index=X.index)

    if not X_nobj.empty:
        if scalar == 0:
            X_num = X_nobj.copy()
        elif scalar == 1:
            scaler = MinMaxScaler()
            X_num = pd.DataFrame(scaler.fit_transform(X_nobj), columns=X_nobj.columns, index=X_nobj.index)
        elif scalar == 2:
            scaler = StandardScaler()
            X_num = pd.DataFrame(scaler.fit_transform(X_nobj), columns=X_nobj.columns, index=X_nobj.index)
    else:
        X_num = pd.DataFrame(index=X.index)

    X_s = pd.concat([X_obj_OH, X_num], axis=1)

    bad_cols = X_s.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    if bad_cols:
        raise ValueError(f"Non-numeric columns remain: {bad_cols}")

    if train_data:
        X_train, X_test, y_train, y_test = train_test_split(
            X_s, y, test_size=0.3, random_state=1, shuffle=True
        )
        return X_train, X_test, y_train, y_test

    return X_s


cols_to_drop = ['High School', 'University', 'Major', 'Weighted GPA', 'GPA', 'Reviewer', 'STEM']

In [86]:
def get_scores(X_train, X_test, y_train, y_test, reg):
    # grid = GridSearchCV(estimator=xgb, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
    reg.fit(X_train, y_train)
    y_prob = reg.predict(X_train)
    train_score = mean_squared_error(y_train, y_prob)
    train_mae = mean_absolute_error(y_train, y_prob)

    y_prob = reg.predict(X_test)
    test_mse = mean_squared_error(y_test, y_prob)
    test_mae = mean_absolute_error(y_test, y_prob)

    return train_score, train_mae, test_mse, test_mae


In [90]:
# Testing all Regressors

df_list = []

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

print(col_combos)

regressor_list = [
    #[LinearSVR(), 'LinearSVR'],
    # [Ridge(), 'Ridge'],
    #[BayesianRidge(), 'BayesianRidge'],
    # [GradientBoostingRegressor(), 'GradientBoostingRegressor'],
    # [PLSRegression(), 'PLSRegression'],
    [SGDRegressor(), 'SGDRegressor'],
    # [ARDRegression(), 'ARDRegression'],
]

regressor_params = {'LinearSVR'                : {'epsilon'          : [0.0, 0.1, 0.5, 1.0, 2.0],
                                                  'tol'              : [0.0001, 0.001, 0.01, 0.00001, 0.000001],
                                                  'C'                : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'loss'             : ['epsilon_insensitive',
                                                                        'squared_epsilon_insensitive'],
                                                  'dual'             : [True, False],
                                                  'fit_intercept'    : [True, False],
                                                  'intercept_scaling': [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'random_state'     : [0], },
                    'Ridge'                    : {'alpha'        : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'fit_intercept': [True, False],
                                                  'max_iter'     : [None],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'solver'       : ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg',
                                                                    'sag', 'saga'],
                                                  'random_state' : [0], },
                    'BayesianRidge'            : {'n_iter'       : [300, 500, 1000],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'      : [0.00001, 0.000001],
                                                  'alpha_2'      : [0.00001, 0.000001],
                                                  'lambda_1'     : [0.00001, 0.000001],
                                                  'lambda_2'     : [0.00001, 0.000001],
                                                  'compute_score': [True, False],
                                                  'fit_intercept': [True, False],
                                                  },
                    'GradientBoostingRegressor': {'loss'                    : ['squared_error', 'absolute_error',
                                                                               'huber', 'quantile'],
                                                  'learning_rate'           : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'n_estimators'            : [100, 200, 500, 1000],
                                                  'criterion'               : ['friedman_mse', 'mse', 'mae'],
                                                  'max_depth'               : [3, 5, 10, 20],
                                                  'min_samples_split'       : [2, 5, 10, 20],
                                                  'min_samples_leaf'        : [1, 2, 5, 10],
                                                  'min_weight_fraction_leaf': [0.0, 0.1, 0.5, 1.0],
                                                  'max_features'            : ['auto', 'sqrt', 'log2'],
                                                  'max_leaf_nodes'          : [None, 10, 20, 50, 100],
                                                  'min_impurity_decrease'   : [0.0, 0.1, 0.5, 1.0],
                                                  'min_impurity_split'      : [None, 0.1, 0.5, 1.0],
                                                  'validation_fraction'     : [0.1, 0.5, 1.0],
                                                  'n_iter_no_change'        : [None, 10, 20, 50, 100],
                                                  'tol'                     : [0.001, 0.01, 0.00001, 0.000001],
                                                  'ccp_alpha'               : [0.0, 0.1, 0.5, 1.0],
                                                  'random_state'            : [0], },
                    'PLSRegression'            : {'n_components': [1, 2, 3, 4, 5, 6, 7, 10, 20],
                                                  'scale'       : [True, False],
                                                  'max_iter'    : [300, 500, 1000, 5000],
                                                  'tol'         : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                                   0.0000001],
                                                  },
                    'ARDRegression'            : {'n_iter'          : [300, 500, 1000],
                                                  'tol'             : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'         : [0.00001, 0.000001],
                                                  'alpha_2'         : [0.00001, 0.000001],
                                                  'lambda_1'        : [0.00001, 0.000001],
                                                  'lambda_2'        : [0.00001, 0.000001],
                                                  'compute_score'   : [True, False],
                                                  'threshold_lambda': [10000.0, 100000.0, 1000000.0],
                                                  'fit_intercept'   : [True, False], },
                    'SGDRegressor'             : {
                        'loss'               : ['squared_error', 'huber', 'epsilon_insensitive',
                                                'squared_epsilon_insensitive'],
                        ## 'penalty'            : ['l2', 'l1', 'elasticnet', None],
                        'alpha'              : [0.001, 0.0001, 0.00001, 0.000001],
                        ## 'l1_ratio'           : [0.15, 0.25, 0.5, 0.75, 1.0],
                        'fit_intercept'      : [True, False],
                        'max_iter'           : [1000, 5000],
                        'tol'                : [0.01, 0.001, 0.0001, 0.00001, 0.000001],
                        # 'epsilon'            : [0.1, 0.01, 0.001, 0.0001],
                        'random_state'       : [0],
                        # 'learning_rate'      : ['constant', 'optimal', 'invscaling', 'adaptive'],
                        # 'eta0'               : [0.1, 0.01, 0.001, 0.0001],
                        # 'power_t'            : [0.5, 0.25, 0.1, 0.01],
                        'early_stopping'     : [True, False],
                        'validation_fraction': [0.1, 0.01],
                        'n_iter_no_change'   : [5, 10, 20, 50, 100],
                        'warm_start'         : [True, False],
                        'average'            : [True, False]
                    }
                    }

for regressor in regressor_list:
    print(regressor[1])
    try:
        for col_to_drop_list in col_combos:
            col_list = list(set(cols_to_drop) - set(col_to_drop_list))
            #print(regressor[1], col_list)
            print("Trying to drop set: ", col_to_drop_list)
            X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACTM_value', col_to_drop_list, debug=False,
                                                         scalar=1, make_graphs=False, train_data=True)
            reg = regressor[0]

            grid = GridSearchCV(estimator=reg, param_grid=regressor_params[regressor[1]], n_jobs=31, cv=10, verbose=3)
            
            grid.fit(X_train, y_train)
            reg = regressor[0]
            reg.set_params(**grid.best_params_)
            train_score, train_mae, test_mse, test_mae = get_scores(X_train, X_test, y_train, y_test, reg)
            print(regressor[1], col_list, train_score, train_mae, test_mse, test_mae)

            df_list.append([regressor[1], col_list, train_score, train_mae, test_mse, test_mae])
    except Exception as e:
        print(e)
        df_list.append([regressor[1], '', -1, -1, -1, -1])
        continue
    df = pd.DataFrame(df_list, columns=['regressor', 'col_list', 'train_score', 'train_mae', 'test_mse', 'test_mae'])
    df.to_csv('regressor_results_ACTMHP_SGD.csv')

[(), ('High School',), ('University',), ('Major',), ('Weighted GPA',), ('GPA',), ('Reviewer',), ('STEM',), ('High School', 'University'), ('High School', 'Major'), ('High School', 'Weighted GPA'), ('High School', 'GPA'), ('High School', 'Reviewer'), ('High School', 'STEM'), ('University', 'Major'), ('University', 'Weighted GPA'), ('University', 'GPA'), ('University', 'Reviewer'), ('University', 'STEM'), ('Major', 'Weighted GPA'), ('Major', 'GPA'), ('Major', 'Reviewer'), ('Major', 'STEM'), ('Weighted GPA', 'GPA'), ('Weighted GPA', 'Reviewer'), ('Weighted GPA', 'STEM'), ('GPA', 'Reviewer'), ('GPA', 'STEM'), ('Reviewer', 'STEM'), ('High School', 'University', 'Major'), ('High School', 'University', 'Weighted GPA'), ('High School', 'University', 'GPA'), ('High School', 'University', 'Reviewer'), ('High School', 'University', 'STEM'), ('High School', 'Major', 'Weighted GPA'), ('High School', 'Major', 'GPA'), ('High School', 'Major', 'Reviewer'), ('High School', 'Major', 'STEM'), ('High Scho

KeyboardInterrupt: 

In [94]:
# Testing all Regressors

df_list = []

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

regressor_list = [  #[LinearSVR(), 'LinearSVR'],
    #[Ridge(), 'Ridge'],
    #[BayesianRidge(), 'BayesianRidge'],
    # [GradientBoostingRegressor(), 'GradientBoostingRegressor'],
    [PLSRegression(), 'PLSRegression'],
    # [ARDRegression(), 'ARDRegression'],
]

regressor_params = {'LinearSVR'                : {'epsilon'          : [0.0, 0.1, 0.5, 1.0, 2.0],
                                                  'tol'              : [0.0001, 0.001, 0.01, 0.00001, 0.000001],
                                                  'C'                : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'loss'             : ['epsilon_insensitive',
                                                                        'squared_epsilon_insensitive'],
                                                  'dual'             : [True, False],
                                                  'fit_intercept'    : [True, False],
                                                  'intercept_scaling': [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'random_state'     : [0], },
                    'Ridge'                    : {'alpha'        : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'fit_intercept': [True, False],
                                                  'max_iter'     : [None],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'solver'       : ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg',
                                                                    'sag', 'saga'],
                                                  'random_state' : [0], },
                    'BayesianRidge'            : {'n_iter'       : [300, 500, 1000],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'      : [0.00001, 0.000001],
                                                  'alpha_2'      : [0.00001, 0.000001],
                                                  'lambda_1'     : [0.00001, 0.000001],
                                                  'lambda_2'     : [0.00001, 0.000001],
                                                  'compute_score': [True, False],
                                                  'fit_intercept': [True, False],
                                                  },
                    'GradientBoostingRegressor': {'loss'                    : ['squared_error', 'absolute_error',
                                                                               'huber', 'quantile'],
                                                  'learning_rate'           : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'n_estimators'            : [100, 200, 500, 1000],
                                                  'criterion'               : ['friedman_mse', 'mse', 'mae'],
                                                  'max_depth'               : [3, 5, 10, 20],
                                                  'min_samples_split'       : [2, 5, 10, 20],
                                                  'min_samples_leaf'        : [1, 2, 5, 10],
                                                  'min_weight_fraction_leaf': [0.0, 0.1, 0.5, 1.0],
                                                  'max_features'            : ['auto', 'sqrt', 'log2'],
                                                  'max_leaf_nodes'          : [None, 10, 20, 50, 100],
                                                  'min_impurity_decrease'   : [0.0, 0.1, 0.5, 1.0],
                                                  'min_impurity_split'      : [None, 0.1, 0.5, 1.0],
                                                  'validation_fraction'     : [0.1, 0.5, 1.0],
                                                  'n_iter_no_change'        : [None, 10, 20, 50, 100],
                                                  'tol'                     : [0.001, 0.01, 0.00001, 0.000001],
                                                  'ccp_alpha'               : [0.0, 0.1, 0.5, 1.0],
                                                  'random_state'            : [0], },
                    'PLSRegression'            : {'n_components': [1, 2, 3, 4, 5, 6, 7, 10, 20],
                                                  'scale'       : [True, False],
                                                  'max_iter'    : [300, 500, 1000, 5000],
                                                  'tol'         : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                                   0.0000001],
                                                  },
                    'ARDRegression'            : {'n_iter'          : [300, 500, 1000],
                                                  'tol'             : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'         : [0.00001, 0.000001],
                                                  'alpha_2'         : [0.00001, 0.000001],
                                                  'lambda_1'        : [0.00001, 0.000001],
                                                  'lambda_2'        : [0.00001, 0.000001],
                                                  'compute_score'   : [True, False],
                                                  'threshold_lambda': [10000.0, 100000.0, 1000000.0],
                                                  'fit_intercept'   : [True, False], }
                    }

for regressor in regressor_list:
    print(regressor[1])
    try:
        for col_to_drop_list in col_combos:
            col_list = list(set(cols_to_drop) - set(col_to_drop_list))
            print(regressor[1], col_list)
            X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACT_value', col_to_drop_list, debug=False,
                                                         scalar=1,
                                                         make_graphs=False, train_data=True)
            reg = regressor[0]

            grid = GridSearchCV(estimator=reg, param_grid=regressor_params[regressor[1]], n_jobs=31, cv=10,
                                verbose=True)
            grid.fit(X_train, y_train)
            reg = regressor[0]
            reg.set_params(**grid.best_params_)
            train_score, train_mae, test_mse, test_mae = get_scores(X_train, X_test, y_train, y_test, reg)
            print(regressor[1], col_list, train_score, train_mae, test_mse, test_mae)

            df_list.append([regressor[1], col_list, train_score, train_mae, test_mse, test_mae])
    except Exception as e:
        print(e)
        df_list.append([regressor[1], '', -1, -1, -1, -1])
        continue
    df = pd.DataFrame(df_list, columns=['regressor', 'col_list', 'train_score', 'train_mae', 'test_mse', 'test_mae'])
    df.to_csv('regressor_results_PLS.csv')

PLSRegression
PLSRegression ['Major', 'Weighted GPA', 'GPA', 'STEM', 'Reviewer', 'High School', 'University']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['Major', 'Weighted GPA', 'GPA', 'STEM', 'Reviewer', 'High School', 'University'] 7.8805969152528235 2.122251733388666 18.62919196192765 3.5770366929874666
PLSRegression ['Weighted GPA', 'Reviewer', 'University', 'GPA', 'STEM', 'Major']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['Weighted GPA', 'Reviewer', 'University', 'GPA', 'STEM', 'Major'] 12.301971481545063 2.6805584217295877 20.814434271250157 3.8279422605414677
PLSRegression ['High School', 'Weighted GPA', 'Reviewer', 'GPA', 'STEM', 'Major']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['High School', 'Weighted GPA', 'Reviewer', 'GPA', 'STEM', 'Major'] 8.785262717364466 2.259146292969171 17.183303939985265 3.3784848932209104
PLSRegression ['High School', 'Weighted GPA', 'Re

C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
2800 fits failed out of a total of 5040.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
560 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCa

PLSRegression ['GPA', 'STEM', 'Reviewer', 'Weighted GPA'] 16.685671464797146 3.2196640555500893 17.518708334601698 3.5663416460349833
PLSRegression ['GPA', 'STEM', 'Reviewer', 'Major']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['GPA', 'STEM', 'Reviewer', 'Major'] 16.164629854828284 3.1992680078296107 21.6863435791882 3.9698324726976275
PLSRegression ['STEM', 'Reviewer', 'Major', 'Weighted GPA']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['STEM', 'Reviewer', 'Major', 'Weighted GPA'] 16.067070557700816 3.1689651420290974 20.848479281487705 3.8598836238643055
PLSRegression ['GPA', 'STEM', 'Major', 'Weighted GPA']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
PLSRegression ['GPA', 'STEM', 'Major', 'Weighted GPA'] 15.457351777986805 3.1059509823851483 20.052519250372953 3.668864610749081
PLSRegression ['GPA', 'Reviewer', 'Major', 'Weighted GPA']
Fitting 10 folds for each of 504 candidates, totalling 

C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
3360 fits failed out of a total of 5040.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
560 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCa

PLSRegression ['GPA', 'STEM', 'Reviewer'] 18.136089018412527 3.4142883350021633 19.748503627794133 3.8540230080617004
PLSRegression ['STEM', 'Reviewer', 'Weighted GPA']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits


...

C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
4480 fits failed out of a total of 5040.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
560 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCa

PLSRegression ['GPA'] 22.610357233596954 3.9435783958672035 27.64589143681687 4.633990030003184
PLSRegression ['Weighted GPA']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits


C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
4480 fits failed out of a total of 5040.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
560 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCa

PLSRegression ['Weighted GPA'] 18.649256557028654 3.59984140685451 20.57807171184054 3.7776708676757176
PLSRegression ['Major']
Fitting 10 folds for each of 504 candidates, totalling 5040 fits
output array is read-only


C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan na

In [32]:
print(1)

1


In [95]:
df = pd.DataFrame(df_list, columns=['regressor', 'col_list', 'train_score', 'train_mae', 'test_mse', 'test_mae'])
print(df)

         regressor                                           col_list  \
0    PLSRegression  [Major, Weighted GPA, GPA, STEM, Reviewer, Hig...   
1    PLSRegression  [Weighted GPA, Reviewer, University, GPA, STEM...   
2    PLSRegression  [High School, Weighted GPA, Reviewer, GPA, STE...   
3    PLSRegression  [High School, Weighted GPA, Reviewer, Universi...   
4    PLSRegression  [High School, Reviewer, University, GPA, STEM,...   
..             ...                                                ...   
120  PLSRegression                                             [STEM]   
121  PLSRegression                                         [Reviewer]   
122  PLSRegression                                              [GPA]   
123  PLSRegression                                     [Weighted GPA]   
124  PLSRegression                                                      

     train_score  train_mae   test_mse  test_mae  
0       7.880597   2.122252  18.629192  3.577037  
1      12.301971   2.

In [96]:
df.to_csv('regressor_results_5.csv')

In [35]:
# Testing all Regressors

df_list = []

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

regressor_list = [[DummyRegressor(), 'DummyRegressor'],
                  [AdaBoostRegressor(), 'AdaBoostRegressor'],
                  [BaggingRegressor(), 'BaggingRegressor'],
                  [ExtraTreesRegressor(), 'ExtraTreesRegressor'],
                  [GradientBoostingRegressor(), 'GradientBoostingRegressor'],
                  [RandomForestRegressor(), 'RandomForestRegressor'],
                  [DecisionTreeRegressor(), 'DecisionTreeRegressor'],
                  [ExtraTreeRegressor(), 'ExtraTreeRegressor'],
                  [KNeighborsRegressor(), 'KNeighborsRegressor'],
                  [RadiusNeighborsRegressor(), 'RadiusNeighborsRegressor'],
                  [LinearSVR(), 'LinearSVR'],
                  [NuSVR(), 'NuSVR'],
                  [SVR(), 'SVR'],
                  [MLPRegressor(), 'MLPRegressor'],
                  [GaussianProcessRegressor(), 'GaussianProcessRegressor'],
                  [KernelRidge(), 'KernelRidge'],
                  [LinearRegression(), 'LinearRegression'],
                  [Ridge(), 'Ridge'],
                  [BayesianRidge(), 'BayesianRidge'],
                  [ARDRegression(), 'ARDRegression'],
                  [Lasso(), 'Lasso'],
                  [LassoLars(), 'LassoLars'],
                  [OrthogonalMatchingPursuit(), 'OrthogonalMatchingPursuit'],
                  [PassiveAggressiveRegressor(), 'PassiveAggressiveRegressor'],
                  [RANSACRegressor(), 'RANSACRegressor'],
                  [SGDRegressor(), 'SGDRegressor'],
                  [TheilSenRegressor(), 'TheilSenRegressor'],
                  [HuberRegressor(), 'HuberRegressor'],
                  [ElasticNet(), 'ElasticNet'],
                  [IsotonicRegression(), 'IsotonicRegression'],
                  [LogisticRegression(), 'LogisticRegression'],
                  [LogisticRegressionCV(), 'LogisticRegressionCV'],
                  # [VotingRegressor(), 'VotingRegressor'],
                  # [StackingRegressor(), 'StackingRegressor'],
                  [TweedieRegressor(), 'TweedieRegressor'],
                  [PoissonRegressor(), 'PoissonRegressor'],
                  [GammaRegressor(), 'GammaRegressor'],
                  [PLSRegression(), 'PLSRegression'],
                  [LGBMRegressor(), 'LGBMRegressor'],
                  # [MultiOutputRegressor(), 'MultiOutputRegressor']
                  ]

for regressor in regressor_list:
    print(regressor[1])
    try:
        for col_list in col_combos:
            X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACTM_value', col_list, debug=False, scalar=1,
                                                         make_graphs=False, train_data=True)
            reg = regressor[0]

            # grid = GridSearchCV(estimator=xgb, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
            train_score, train_mae, test_mse, test_mae = get_scores(X_train, X_test, y_train, y_test, reg)

            df_list.append([regressor[1], col_list, train_score, train_mae, test_mse, test_mae])
    except Exception as e:
        df_list.append([regressor[1], '', -1, -1, -1, -1])
        continue


DummyRegressor
AdaBoostRegressor
BaggingRegressor
ExtraTreesRegressor
GradientBoostingRegressor
RandomForestRegressor
DecisionTreeRegressor
ExtraTreeRegressor
KNeighborsRegressor
RadiusNeighborsRegressor
LinearSVR
NuSVR
SVR
MLPRegressor
GaussianProcessRegressor
KernelRidge
LinearRegression
Ridge
BayesianRidge
ARDRegression
Lasso
LassoLars
OrthogonalMatchingPursuit
PassiveAggressiveRegressor
RANSACRegressor
SGDRegressor
TheilSenRegressor
HuberRegressor
ElasticNet
IsotonicRegression
LogisticRegression
LogisticRegressionCV
TweedieRegressor
PoissonRegressor
GammaRegressor
PLSRegression
LGBMRegressor


C:\Users\Cristian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveRegressor is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDRegressor(loss='epsilon_insensitive', penalty=None, learning_rate='pa1', eta0 = 1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
# Testing all Regressors

df_list = []

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

regressor_list = [
    #[LinearSVR(), 'LinearSVR'],
    # [Ridge(), 'Ridge'],
    #[BayesianRidge(), 'BayesianRidge'],
    # [GradientBoostingRegressor(), 'GradientBoostingRegressor'],
    # [PLSRegression(), 'PLSRegression'],
    [SGDRegressor(), 'SGDRegressor'],
    # [ARDRegression(), 'ARDRegression'],
]

regressor_params = {'LinearSVR'                : {'epsilon'          : [0.0, 0.1, 0.5, 1.0, 2.0],
                                                  'tol'              : [0.0001, 0.001, 0.01, 0.00001, 0.000001],
                                                  'C'                : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'loss'             : ['epsilon_insensitive',
                                                                        'squared_epsilon_insensitive'],
                                                  'dual'             : [True, False],
                                                  'fit_intercept'    : [True, False],
                                                  'intercept_scaling': [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'random_state'     : [0], },
                    'Ridge'                    : {'alpha'        : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'fit_intercept': [True, False],
                                                  'max_iter'     : [None],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'solver'       : ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg',
                                                                    'sag', 'saga'],
                                                  'random_state' : [0], },
                    'BayesianRidge'            : {'n_iter'       : [300, 500, 1000],
                                                  'tol'          : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'      : [0.00001, 0.000001],
                                                  'alpha_2'      : [0.00001, 0.000001],
                                                  'lambda_1'     : [0.00001, 0.000001],
                                                  'lambda_2'     : [0.00001, 0.000001],
                                                  'compute_score': [True, False],
                                                  'fit_intercept': [True, False],
                                                  },
                    'GradientBoostingRegressor': {'loss'                    : ['squared_error', 'absolute_error',
                                                                               'huber', 'quantile'],
                                                  'learning_rate'           : [0.1, 0.5, 1.0, 2.0, 5.0],
                                                  'n_estimators'            : [100, 200, 500, 1000],
                                                  'criterion'               : ['friedman_mse', 'mse', 'mae'],
                                                  'max_depth'               : [3, 5, 10, 20],
                                                  'min_samples_split'       : [2, 5, 10, 20],
                                                  'min_samples_leaf'        : [1, 2, 5, 10],
                                                  'min_weight_fraction_leaf': [0.0, 0.1, 0.5, 1.0],
                                                  'max_features'            : ['auto', 'sqrt', 'log2'],
                                                  'max_leaf_nodes'          : [None, 10, 20, 50, 100],
                                                  'min_impurity_decrease'   : [0.0, 0.1, 0.5, 1.0],
                                                  'min_impurity_split'      : [None, 0.1, 0.5, 1.0],
                                                  'validation_fraction'     : [0.1, 0.5, 1.0],
                                                  'n_iter_no_change'        : [None, 10, 20, 50, 100],
                                                  'tol'                     : [0.001, 0.01, 0.00001, 0.000001],
                                                  'ccp_alpha'               : [0.0, 0.1, 0.5, 1.0],
                                                  'random_state'            : [0], },
                    'PLSRegression'            : {'n_components': [1, 2, 3, 4, 5, 6, 7, 10, 20],
                                                  'scale'       : [True, False],
                                                  'max_iter'    : [300, 500, 1000, 5000],
                                                  'tol'         : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                                   0.0000001],
                                                  },
                    'ARDRegression'            : {'n_iter'          : [300, 500, 1000],
                                                  'tol'             : [0.001, 0.01, 0.00001, 0.000001],
                                                  'alpha_1'         : [0.00001, 0.000001],
                                                  'alpha_2'         : [0.00001, 0.000001],
                                                  'lambda_1'        : [0.00001, 0.000001],
                                                  'lambda_2'        : [0.00001, 0.000001],
                                                  'compute_score'   : [True, False],
                                                  'threshold_lambda': [10000.0, 100000.0, 1000000.0],
                                                  'fit_intercept'   : [True, False], },
                    'SGDRegressor'             : {
                        'loss'               : ['squared_error'],
                        #, 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'],
                        'penalty'            : ['l2', 'l1', 'elasticnet', None],
                        'alpha'              : [0.00001, 0.000001],
                        'l1_ratio'           : [0.15, 0.25, 0.5, 0.75, 1.0],
                        'fit_intercept'      : [True, False],
                        'max_iter'           : [1000, 5000, 10000],
                        'tol'                : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                0.0000001],
                        'epsilon'            : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                0.0000001],
                        'random_state'       : [0],
                        'learning_rate'      : ['constant', 'optimal', 'invscaling', 'adaptive'],
                        'eta0'               : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001,
                                                0.0000001],
                        'power_t'            : [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001],
                        'early_stopping'     : [True, False],
                        'validation_fraction': [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001],
                        'n_iter_no_change'   : [5, 10, 20, 50, 100],
                        'warm_start'         : [True, False],
                        'average'            : [True, False]
                    }
                    }

for regressor in regressor_list:
    print(regressor[1])
    try:
        for col_to_drop_list in col_combos:
            col_list = list(set(cols_to_drop) - set(col_to_drop_list))
            print(regressor[1], col_list)
            X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACTM_value', col_to_drop_list, debug=False,
                                                         scalar=1, make_graphs=False, train_data=True)
            reg = regressor[0]

            grid = GridSearchCV(estimator=reg, param_grid=regressor_params[regressor[1]], n_jobs=31, cv=10,
                                verbose=True)
            grid.fit(X_train, y_train) 
            reg = regressor[0]
            reg.set_params(**grid.best_params_)
            train_score, train_mae, test_mse, test_mae = get_scores(X_train, X_test, y_train, y_test, reg)
            print(regressor[1], col_list, train_score, train_mae, test_mse, test_mae)

            df_list.append([regressor[1], col_list, train_score, train_mae, test_mse, test_mae])
    except Exception as e:
        print(e)
        df_list.append([regressor[1], '', -1, -1, -1, -1])
        continue
    df = pd.DataFrame(df_list, columns=['regressor', 'col_list', 'train_score', 'train_mae', 'test_mse', 'test_mae'])
    df.to_csv('regressor_results_ACTMHP_SGD.csv')

SGDRegressor
SGDRegressor ['Major', 'Weighted GPA', 'GPA', 'STEM', 'Reviewer', 'High School', 'University']
could not convert string to float: 'Lindblom Math & Science Academy'


In [12]:
df = pd.DataFrame(df_list, columns=['regressor', 'col_list', 'train_score', 'train_mae', 'test_mse', 'test_mae'])
df.to_csv('regressor_results_ACTM.csv')

In [28]:
# ACT for Randomized Search, 1000 iterations

col_combos = list()
for r in range(len(cols_to_drop)):
    #col_list = list(set(cols_to_drop) - set(col_to_drop_list))
    col_combos += list(combinations(cols_to_drop, r))

    #print(col_combos)

for col_list in col_combos:
    print("JERE")
    X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACT_value', col_list, debug=False, scalar=1,
                                                 make_graphs=False, train_data=True)
    params_gs = {
        'min_child_weight': [0, 0.5, 1, 5, 10],
        'eta'             : [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        'gamma'           : [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 10, 100],
        'lambda'          : [0, 0.1, 0.5, 1.0],
        'alpha'           : [0, 0.1, 0.5, 1.0],
        'subsample'       : [0.1, 0.25, 0.5, 0.75, 1],
        'max_depth'       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'n_estimators'    : [1, 2, 3, 4, 5, 10, 50, 100],
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    params_rs = {
        'min_child_weight': uniform(0, 10),
        'eta'             : uniform(0, 0.75),
        'gamma'           : uniform(0, 100),
        'lambda'          : uniform(0, 1),
        'alpha'           : uniform(0, 1),
        'subsample'       : uniform(0, 1),
        'max_depth'       : randint(1, 10),
        'n_estimators'    : randint(1, 100),
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    xgb = XGBRegressor()

    # grid = GridSearchCV(estimator=xgb, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
    grid = RandomizedSearchCV(estimator=xgb, param_distributions=params_rs, n_iter=1000, n_jobs=31, cv=30, verbose=True)
    grid.fit(X_train, y_train)
    print(grid.best_params_)
    y_prob = grid.best_estimator_.predict(X_train)
    train_score = mean_squared_error(y_train, y_prob)
    train_mae = mean_absolute_error(y_train, y_prob)
    print(f'Train MSE: {round(train_score, 2)} - MAE: {round(train_mae, 2)}')

    y_prob = grid.best_estimator_.predict(X_test)
    test_mse = mean_squared_error(y_test, y_prob)
    test_mae = mean_absolute_error(y_test, y_prob)
    print(f'Test  MSE: {round(test_mse, 2)} - MAE: {round(test_mae, 2)}')

JERE


ValueError: could not convert string to float: 'Lindblom Math & Science Academy'

In [ ]:
# ACT for Randomized Search, 1000 iterations

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

for col_list in col_combos:
    print(col_list)
    X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACT_value', col_list, debug=False, scalar=1,
                                                 make_graphs=False, train_data=True)
    params_gs = {
        'min_child_weight': [0, 0.5, 1, 5, 10],
        'eta'             : [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        'gamma'           : [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 10, 100],
        'lambda'          : [0, 0.1, 0.5, 1.0],
        'alpha'           : [0, 0.1, 0.5, 1.0],
        'subsample'       : [0.1, 0.25, 0.5, 0.75, 1],
        'max_depth'       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'n_estimators'    : [1, 2, 3, 4, 5, 10, 50, 100],
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    params_rs = {
        'min_child_weight': uniform(0, 10),
        'eta'             : uniform(0, 0.75),
        'gamma'           : uniform(0, 100),
        'lambda'          : uniform(0, 1),
        'alpha'           : uniform(0, 1),
        'subsample'       : uniform(0, 1),
        'max_depth'       : randint(1, 10),
        'n_estimators'    : randint(1, 100),
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    xgb = XGBRegressor()

    # grid = GridSearchCV(estimator=xgb, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
    grid = RandomizedSearchCV(estimator=xgb, param_distributions=params_rs, n_iter=1000, n_jobs=31, cv=30, verbose=True)
    grid.fit(X_train, y_train)
    print(grid.best_params_)
    y_prob = grid.best_estimator_.predict(X_train)
    train_score = mean_squared_error(y_train, y_prob)
    train_mae = mean_absolute_error(y_train, y_prob)
    print(f'Train MSE: {round(train_score, 2)} - MAE: {round(train_mae, 2)}')

    y_prob = grid.best_estimator_.predict(X_test)
    test_mse = mean_squared_error(y_test, y_prob)
    test_mae = mean_absolute_error(y_test, y_prob)
    print(f'Test  MSE: {round(test_mse, 2)} - MAE: {round(test_mae, 2)}')

()
Fitting 30 folds for each of 1000 candidates, totalling 30000 fits
{'alpha': np.float64(0.2536876191893298), 'eta': np.float64(0.5774482092099994), 'gamma': np.float64(45.26965489190322), 'lambda': np.float64(0.8509370029455791), 'max_depth': 2, 'min_child_weight': np.float64(4.0498397693489405), 'n_estimators': 2, 'n_jobs': 1, 'random_state': 1, 'subsample': np.float64(0.7671305271209514)}
Train MSE: 15.86 - MAE: 3.23
Test  MSE: 19.51 - MAE: 3.69
('High School',)
Fitting 30 folds for each of 1000 candidates, totalling 30000 fits
{'alpha': np.float64(0.3326228015525028), 'eta': np.float64(0.41476904228408384), 'gamma': np.float64(92.59955305364393), 'lambda': np.float64(0.9868935872748685), 'max_depth': 7, 'min_child_weight': np.float64(5.050420022673167), 'n_estimators': 11, 'n_jobs': 1, 'random_state': 1, 'subsample': np.float64(0.9409844946480592)}
Train MSE: 13.02 - MAE: 2.89
Test  MSE: 15.07 - MAE: 3.24
('University',)
Fitting 30 folds for each of 1000 candidates, totalling 300

KeyboardInterrupt: 

In [92]:
# ACT for Grid Search

cols_to_drop = ['High School', 'University', 'Major', 'Weighted GPA', 'GPA', 'Reviewer', 'STEM']

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

for col_list in col_combos:
    print(col_list)
    X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACT_value', col_list, debug=False, scalar=1,
                                                 make_graphs=False, train_data=True)
    params_gs = {
        'min_child_weight': [0, 0.5, 1, 5, 10],
        'eta'             : [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        'gamma'           : [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 10, 100],
        'lambda'          : [0, 0.1, 0.5, 1.0],
        'alpha'           : [0, 0.1, 0.5, 1.0],
        'subsample'       : [0.1, 0.25, 0.5, 0.75, 1],
        'max_depth'       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'n_estimators'    : [1, 2, 3, 4, 5, 10, 50, 100],
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    params_rs = {
        'min_child_weight': uniform(0, 10),
        'eta'             : uniform(0, 0.75),
        'gamma'           : uniform(0, 100),
        'lambda'          : uniform(0, 1),
        'alpha'           : uniform(0, 1),
        'subsample'       : uniform(0, 1),
        'max_depth'       : randint(1, 10),
        'n_estimators'    : randint(1, 100),
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    grid = XGBRegressor()

    grid = GridSearchCV(estimator=grid, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
    # grid = RandomizedSearchCV(estimator=xgb, param_distributions=params_rs, n_iter=1000, n_jobs=31, cv=30, verbose=True)
    grid.fit(X_train, y_train)
    print(grid.best_params_)
    y_prob = grid.best_estimator_.predict(X_train)
    # y_prob = grid.predict(X_train)
    train_score = mean_squared_error(y_train, y_prob)
    train_mae = mean_absolute_error(y_train, y_prob)
    print(f'Train MSE: {round(train_score, 2)} - MAE: {round(train_mae, 2)}')

    y_prob = grid.best_estimator_.predict(X_test)
    # y_prob = grid.predict(X_test)
    test_mse = mean_squared_error(y_test, y_prob)
    test_mae = mean_absolute_error(y_test, y_prob)
    print(f'Test  MSE: {round(test_mse, 2)} - MAE: {round(test_mae, 2)}')

()
Fitting 30 folds for each of 1728000 candidates, totalling 51840000 fits


KeyboardInterrupt: 

In [46]:
# ACTM for Grid Search

cols_to_drop = ['High School', 'University', 'Major', 'Weighted GPA', 'GPA', 'Reviewer', 'STEM']

col_combos = list()
for r in range(len(cols_to_drop)):
    col_combos += list(combinations(cols_to_drop, r))

for col_list in col_combos:
    print(col_list)
    X_train, X_test, y_train, y_test = data_load('test_data.csv', 'ACTM_value', col_list, debug=False, scalar=1,
                                                 make_graphs=False, train_data=True)
    params_gs = {
        # 'min_child_weight': [0, 0.5, 1, 5, 10],
        'eta'         : [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        # 'gamma'           : [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 10, 100],
        # 'lambda'          : [0, 0.1, 0.5, 1.0],
        'alpha'       : [0, 0.1, 0.5, 1.0],
        # 'subsample'       : [0.1, 0.25, 0.5, 0.75, 1],
        'max_depth'   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'n_estimators': [1, 2, 3, 4, 5, 10, 50, 100],
        'random_state': [1],
        'n_jobs'      : [1]
    }

    params_rs = {
        'min_child_weight': uniform(0, 10),
        'eta'             : uniform(0, 0.75),
        'gamma'           : uniform(0, 100),
        'lambda'          : uniform(0, 1),
        'alpha'           : uniform(0, 1),
        'subsample'       : uniform(0, 1),
        'max_depth'       : randint(1, 10),
        'n_estimators'    : randint(1, 100),
        'random_state'    : [1],
        'n_jobs'          : [1]
    }

    grid_m = XGBRegressor()

    grid_m = GridSearchCV(estimator=grid_m, param_grid=params_gs, n_jobs=31, cv=30, verbose=True)
    # grid = RandomizedSearchCV(estimator=xgb, param_distributions=params_rs, n_iter=1000, n_jobs=31, cv=30, verbose=True)
    grid_m.fit(X_train, y_train)
    print(grid_m.best_params_)
    y_prob = grid_m.best_estimator_.predict(X_train)
    # y_prob = grid.predict(X_train)
    train_score = mean_squared_error(y_train, y_prob)
    train_mae = mean_absolute_error(y_train, y_prob)
    print(f'Train MSE: {round(train_score, 2)} - MAE: {round(train_mae, 2)}')

    y_prob = grid_m.best_estimator_.predict(X_test)
    # y_prob = grid.predict(X_test)
    test_mse = mean_squared_error(y_test, y_prob)
    test_mae = mean_absolute_error(y_test, y_prob)
    print(f'Test  MSE: {round(test_mse, 2)} - MAE: {round(test_mae, 2)}')

()


ValueError: could not convert string to float: 'Lindblom Math & Science Academy'

In [ ]:
cols_to_drop = ['Major', 'GPA']

X_real_test = data_load('real_test_data.csv', 'ACTM_value', cols_to_drop=cols_to_drop, debug=False, scalar=1, train_data=False)
X_real_test = X_real_test.reindex(columns=grid.best_estimator_.feature_names_in_, fill_value=0)

y_prob = grid.best_estimator_.predict(X_real_test)
print(y_prob)
#plot_importance(grid.best_estimator_, max_num_features=10)  # top 10 most important features

[27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511 27.19146511
 27.19146511 27.19146511 27.19146511 27.19146511 27

ValueError: tree must be Booster, XGBModel or dict instance

In [105]:
cols_to_drop = ['Major', 'Weighted GPA', 'STEM']

X_real_test = data_load('real_test_data.csv', 'ACTM_value', cols_to_drop=cols_to_drop, debug=False, scalar=1, train_data=False)

y_prob = grid_m.best_estimator_.predict(X_real_test)
print(y_prob)
plot_importance(grid_m.best_estimator_, max_num_features=10)  # top 10 most important features


NameError: name 'grid_m' is not defined